In [ ]:
import sys, subprocess, os, shutil, glob
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap", "huggingface_hub"], check=True)
from pycocoevalcap.spice import spice
libs = Path(spice.__file__).resolve().parent / "lib"; libs.mkdir(exist_ok=True)
for name, size in {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}.items():
    have = libs / name
    if not have.is_file() or have.stat().st_size != size:
        c = [p for p in Path("/kaggle/input").rglob(name) if p.stat().st_size == size]
        assert c, "thiếu " + name
        shutil.copy2(c[0], have)
v = subprocess.run(["bash", "-c", "java -version 2>&1"], capture_output=True, text=True).stdout
if 'version "1.8' not in v:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-8-jre-headless", shell=True, check=True)
    j = sorted(glob.glob("/usr/lib/jvm/*8*")); assert j, "không có Java 8"
    os.environ["JAVA_HOME"] = j[0]; os.environ["PATH"] = j[0] + "/bin:" + os.environ["PATH"]
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[:1], flush=True)


In [ ]:
import sys, subprocess, shutil, time, json
from pathlib import Path
from huggingface_hub import hf_hub_download
forest = hf_hub_download("HarrytheOrange/parsed_AndroidControl", "all_forest_dict.zip", repo_type="dataset")
d = [p.parent for p in Path("/kaggle/input").rglob("gate_b_rgtc.py")]
assert len(d) == 1, d
d = d[0]; w = Path("/kaggle/working"); out = w / "gate_b"
shutil.copy2(d / "gate_b_rgtc.py", w / "gate_b_rgtc.py")
assert subprocess.run([sys.executable, str(w / "gate_b_rgtc.py"), "--selftest"]).returncode == 0
log = w / "gate_b.log"
with log.open("w") as f:
    p = subprocess.Popen([sys.executable, "-u", str(w / "gate_b_rgtc.py"), "--root", str(d), "--forest", forest,
                          "--exclude", str(d / "excl_ep.json"), "--out", str(out)], stdout=f, stderr=subprocess.STDOUT, text=True)
    while p.poll() is None:
        time.sleep(120)
        print(time.strftime("%H:%M"), log.read_text(errors="replace")[-180:].replace("\n", " | "), flush=True)
print(log.read_text(errors="replace")[-9000:])
assert p.returncode == 0, p.returncode
r = json.loads((out / "gate_b_rgtc.json").read_text())
assert r["pool_steps"] == 49046 and r["pool_episodes"] == 10726, (r["pool_steps"], r["pool_episodes"])
shutil.make_archive(str(w / "gate_b_ket_qua"), "zip", str(out))
print("XONG")
